In [1]:
from __future__ import annotations

import pathlib
from collections import defaultdict

import numpy as np
import pandas as pd
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [2]:
dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Fragments Resolved"
    # / "sample"
    / "rebel_entity_fragments.jsonl"
)

Load the fragments

In [3]:
fragments = []

# we will not be using fragments with no names
fragments_with_no_names = 0

# load the data
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fragment = ResolvedWikidataEntity.from_json(line.strip())

        if not fragment.names:
            fragments_with_no_names += 1
            del fragment
            continue

        # reduce memory footprint
        if "doc_id" in fragment.metadata:
            del fragment.metadata["doc_id"]

        if "source_text_hash" in fragment.metadata:
            del fragment.metadata["source_text_hash"]

        if "fragment_id" in fragment.metadata:
            del fragment.metadata["fragment_id"]

        fragment.evidence_map = None
        fragment.source_ids = None

        fragments.append(fragment)

print(f"Loaded {len(fragments):,d} fragments, ignored {fragments_with_no_names:,d} fragments with no names")

Loaded 7,116,843 fragments, ignored 0 fragments with no names


Example fragment

In [4]:
fragments[0]

ResolvedWikidataEntity(entity_id='Q5395951', properties=defaultdict(<class 'list'>, {'name': ['Erul Heights']}), source_ids=None, evidence_map=None, metadata={'type': ['upland']})

In [5]:
# compute counts of property occurrence and entity types of the fragments
property_counts = defaultdict(int)
type_counts = defaultdict(int)

for fragment in fragments:
    for prop_name, prop_value in fragment.properties.items():
        if prop_value:
            property_counts[prop_name] += 1

    for ent_type in fragment.wikidata_type:
        type_counts[ent_type] += 1

Top properties by occurrence in the fragments
---

In [6]:
df = (
    pd.DataFrame(property_counts.items(), columns=["property", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)
df.to_csv("property_occurrence.csv", index=False)
df[:20]

,property,count
0,name,7116843
1,located in the administrative territorial entity,662301
2,date of birth,653842
3,country,472135
4,instance of,308019
5,date of death,265785
6,sport,258965
7,publication date,171695
8,point in time,167143
9,place of birth,158241


Top entity types of the fragments
---

In [7]:
df = (
    pd.DataFrame(type_counts.items(), columns=["type", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)
df.to_csv("type_occurrence.csv", index=False)
df[:20]

,type,count
0,human,1701207
1,taxon,284566
2,film,151307
3,album,122653
4,corporation,108961
5,sports series,106074
6,village,87529
7,human settlement,83840
8,musical group,66295
9,literary work,59003


Properties overlap
---

In [8]:
# for each property how often that two distinct fragments have (1) a value for this property, and (2) the same value this property
property_value_counts = defaultdict(lambda: defaultdict(int))
entity_value_counts = defaultdict(int)

for entity in fragments:
    for prop_name, values in entity.properties.items():
        entity_value_counts[prop_name] += 1
        for value in values:
            property_value_counts[prop_name][value] += 1

entity_count = len(fragments)

rows = []
for prop_name, value_counts in property_value_counts.items():
    arr = np.array(list(value_counts.values()))
    ent_val_count = entity_value_counts[prop_name]
    ent_probs = arr / entity_count
    cond_ent_probs = arr / ent_val_count
    prob = (ent_probs**2).sum()
    cond_prob = (cond_ent_probs**2).sum()
    ent_fraction = ent_val_count / entity_count
    rows.append((prop_name, len(value_counts), prob, cond_prob, ent_fraction))

overlap_df = pd.DataFrame(
    rows, columns=["property", "distinct_value_count", "overlap_prob", "cond_overlap_prob", "entities_fraction"]
)
overlap_df = overlap_df.sort_values("overlap_prob", ascending=False)
overlap_df.head(100)

,property,distinct_value_count,overlap_prob,cond_overlap_prob,entities_fraction
21,sport,477,1.599747e-04,0.120821,0.036388
8,country,855,1.062984e-04,0.024153,0.066341
14,instance of,11930,1.193202e-05,0.006370,0.043280
9,located in the administrative territorial entity,55765,6.325941e-06,0.000730,0.093061
23,point in time,14742,5.000785e-06,0.009066,0.023486
...,...,...,...,...,...
208,participating team,1331,4.959599e-09,0.007841,0.000795
238,basin country,165,4.929846e-09,0.031713,0.000394
122,winner,7032,4.584057e-09,0.000661,0.002633
369,taxon rank,13,4.574482e-09,0.404320,0.000106
